# Why convexity?

In the previous notebooks we established what convexity *is*. In this notebook, we'll talk about why we care about it, and we'll do it by writing our first convex program.

There are three payoffs when it comes to convex optimization. We'll see all three on problems today, so we can focus on the main ideas for now:

1. The solution is a property of the *problem*, not the initialization, the step size, or the method we picked.
2. We don't pick the solver, CVXPY does. A whole ecosystem of solvers sits behind a single call.
3. Those same solvers double as an oracle we can use to debug any custom code or algorithm we develop.

## Our first convex program

We'll use [CVXPY](https://www.cvxpy.org/), a Python library that lets us write a convex problem in math notation and have it solved for us. Think of it as the LaTeX of optimization. We describe the problem at a high level, and a backend takes care of the details.

The example below is a small quadratic program with box constraints from Boyd and Vandenberghe's textbook,
$$ \min_x \tfrac{1}{2} x^\top P x + q^\top x + r \quad \text{subject to} \quad -1 \le x_i \le 1, $$
with $P \succeq 0$. Read the next cell as math, not as code. That is kind of the point of CVXPY.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path('..').resolve()))
sys.path.append(str(pathlib.Path('../render_utility').resolve()))

import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
from definitions import blue_2, red_2, orange_2, green_2, grey_1

P = np.array([[13.0, 12.0, -2.0],
              [12.0, 17.0,  6.0],
              [-2.0,  6.0, 12.0]])
q = np.array([-22.0, -14.5, 13.0])
r = 1.0

x = cp.Variable(3)
objective = cp.Minimize(0.5 * cp.quad_form(x, P) + q @ x + r)
constraints = [x >= -1, x <= 1]
problem = cp.Problem(objective, constraints)

problem.solve()
print('Optimal value:', problem.value)
print('Optimal x:', x.value)

The textbook says the answer is $-21.625$ at $x^* = (1, \tfrac{1}{2}, -1)$. We just got that in six lines, and we never had to pick a step size, an initial point, or a stopping criterion. The reason we *could* skip all of those decisions is that the problem is convex. CVXPY knows how to make those choices on our behalf.

If we changed one of the entries of $P$ so the matrix is no longer positive semidefinite (for example, by flipping a sign on the diagonal), CVXPY would refuse to solve the problem and tell us why. That refusal is a feature: it catches the fact that the problem is no longer convex before we waste time on it.

## The solution is a property of the problem

A convex problem has the property that *every* local minimum is also the global minimum. Run any reasonable descent method, from any starting point, and you'll land on the same answer. This is a *structural* property of the problem, not a feature of any one solver.

To see what it buys us, we'll run the same gradient descent code on two problems, one convex and one not, and look at where it lands from a hundred random starts. The two problems are the parabola and the double well from notebook 2.

In [ ]:
def pgd_1d(grad, x0, lo, hi, n_iters=500, step=0.02):
    x = x0
    for _ in range(n_iters):
        x = np.clip(x - step * grad(x), lo, hi)
    return x

grad_convex = lambda x: 2 * x
grad_double = lambda x: 4 * x * (x**2 - 1)

rng = np.random.default_rng(0)
starts = rng.uniform(-2, 2, size=100)
finals_convex = np.array([pgd_1d(grad_convex, x0, -2, 2) for x0 in starts])
finals_double = np.array([pgd_1d(grad_double, x0, -2, 2) for x0 in starts])

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].hist(finals_convex, bins=30, range=(-2, 2),
             color=blue_2, alpha=0.7, edgecolor=blue_2, linewidth=2)
axes[0].set_title('$f(x) = x^2$ (convex)')
axes[0].set_xlabel('x'); axes[0].set_ylabel('count')

axes[1].hist(finals_double, bins=30, range=(-2, 2),
             color=red_2, alpha=0.7, edgecolor=red_2, linewidth=2)
axes[1].set_title('$f(x) = (x^2 - 1)^2$ (non-convex)')
axes[1].set_xlabel('x'); axes[1].set_ylabel('count')
plt.tight_layout()
plt.show()

On the left, all hundred starts collapse to a single value at $x = 0$. On the right, they split into two piles at $x = \pm 1$, the two well locations. The PGD code is *identical* in both runs. The only thing that changed is the problem.

That's the structural guarantee we're buying. With a convex problem we don't have to think about initialization, restarts, or seeding. With a non-convex one we are stuck running multiple starts and hoping one of them found the right basin.

## We don't have to pick the solver, CVXPY does

Convex optimization has a number of backend solvers behind it (Mosek, Clarabel, SCS, OSQP, ECOS, Gurobi, ...). We never have to interact with any of them directly. We write the problem in CVXPY and call `.solve()`. CVXPY translates our model into a standard cone form, looks at which backends are installed, and dispatches to whichever one it deems best for the problem class. The whole solver ecosystem sits behind a single call.

In [ ]:
print('installed solvers:', cp.installed_solvers())
problem.solve()
print('CVXPY chose:', problem.solver_stats.solver_name)
print('value:', problem.value)

That's the entire interface. We didn't import a solver, didn't pick one, didn't pass a tolerance or a step size. CVXPY did all of that on our behalf based on the shape of the problem.

The SDP and SOCP examples in Part II benefit a lot from Mosek, which is the gold standard for those classes and free for academic use.

## Convex solvers as a debugging oracle

Those same solvers double as an oracle we can use to debug any custom code or algorithm we develop. CVXPY is slower than a hand-written solver, but it is correct by construction. So whenever we are writing a custom ADMM, or any other piece of code we want to be both fast and trustworthy for convex optimization, we can build the same problem in CVXPY at small scale and use it as ground truth.

To make this concrete, the function below contains a deliberate sign error in its gradient. See if you can spot it before you compare to CVXPY.

In [ ]:
def buggy_pgd(P, q, n_iters=2000, step=0.02):
    x = np.zeros(P.shape[0])
    for _ in range(n_iters):
        grad = P @ x - q   # incorrect sign on q
        x = np.clip(x - step * grad, -1, 1)
    return x

x_bug = buggy_pgd(P, q)
val_bug = 0.5 * x_bug @ P @ x_bug + q @ x_bug + r

print('CVXPY x:', x.value)
print('buggy PGD x:', x_bug)
print('CVXPY value:', problem.value)
print('buggy PGD value:', val_bug)

The two solutions are visibly different and the buggy objective sits well above CVXPY's. Without the CVXPY reference, we would have no easy way to tell whether the custom solver was almost right or wildly broken. The bug, by the way, is a sign error: the gradient of $\tfrac{1}{2} x^\top P x + q^\top x$ is $P x + q$, not $P x - q$.

Once we fix the sign and rerun, the two values match to many digits and the recovered $x$ matches CVXPY's. We will come back to this workflow in Part II whenever we hand-write a relaxation.

## Summary

Convexity facilitates our workflow when dealing with optimization. We can delegate the choice of step size, the initialization, the convergence check, and even the correctness of our own custom code, all to an off-the-shelf solver.

In the next notebook we learn about standard classes of convex problems: LP, QP, QCQP, SOCP, and SDP.

## References

[1] *Convex Optimization*. Boyd, S.; Vandenberghe, L.
       Cambridge University Press (2004).
       :DOI:`10.1017/CBO9780511804441`

[2] *CVXPY: A Python-embedded modeling language for convex optimization*.
       Diamond, S.; Boyd, S.
       Journal of Machine Learning Research 17 (83): 1 (2016).